This post is a Jupyter notebook (`index.ipynb`). It is also a check that notebook posts render
on the website: if you can see the chart below, it works.

We use NOAA's [Climate at a Glance](https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/)
statewide time series: total precipitation in California for each calendar year, averaged over the state.
No login is needed.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"  # interactive charts that also work on the website

URL = (
    "https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/"
    "4/pcp/12/12/1895-2025/data.csv"  # 4 = California, pcp = precipitation, 12/12 = January-December
)
df = pd.read_csv(URL, comment="#")
df["Year"] = df["Date"] // 100
df = df.rename(columns={"Value": "Precipitation (in)"})[["Year", "Precipitation (in)"]]
df.tail()

,Year,Precipitation (in)
126,2021,18.93
127,2022,14.10
128,2023,27.38
129,2024,26.09
130,2025,23.29


## Wettest and driest years

In [2]:
mean = df["Precipitation (in)"].mean()
print(f"Average 1895-2025: {mean:.1f} inches per year")
pd.concat([
    df.nlargest(3, "Precipitation (in)").assign(Rank="Wettest"),
    df.nsmallest(3, "Precipitation (in)").assign(Rank="Driest"),
]).set_index("Rank")

Average 1895-2025: 22.1 inches per year


,Year,Precipitation (in)
Rank,,
Wettest,1983,42.46
Wettest,1998,36.41
Wettest,1995,35.20
Driest,2013,7.93
Driest,1976,11.75
Driest,2020,12.07


## Yearly precipitation

The line shows the 10-year rolling average, which makes long dry periods (for example 2012-2016) easier to see.

In [3]:
df["10-year average"] = df["Precipitation (in)"].rolling(10).mean()

fig = px.bar(df, x="Year", y="Precipitation (in)", title="California yearly precipitation (NOAA)")
fig.update_traces(name="Yearly total", showlegend=True)
fig.add_scatter(x=df["Year"], y=df["10-year average"], mode="lines", name="10-year average")
fig.add_hline(y=mean, line_dash="dot", annotation_text=f"1895-2025 average: {mean:.1f} in",
              annotation_position="top left", annotation_bgcolor="white")
fig.update_layout(height=420, legend=dict(orientation="h", y=-0.2))
fig.show()

*Data: NOAA National Centers for Environmental Information, Climate at a Glance: Statewide Time Series.*